<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Week8/Day4/Exercices_XP/Exercises_MCP_LLM_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises (Student) - MCP Client with LLM

In [ ]:
!pip install -q mcp nest_asyncio requests

In [ ]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [89]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@mcp.tool()
def greet(name: str) -> str:
    """Return a greeting string."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()

Overwriting server.py


## Exercise 1 (provide answer)

STDIO transport is simpler for local development because it uses standard input/output streams of a child process. It avoids the overhead of managing network ports, handling HTTP authentication/CORS, and managing a long-running server process independently of the client.

## Exercise 2

In [86]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    # Use python to run the server script
    params = StdioServerParameters(command="python", args=["server.py"], env=None)
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            print("Connection initialized successfully")

In [ ]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


Exercise 2: OK (connected and initialized)


## Exercise 3

In [87]:
async def ex3_list():
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            # List resources
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            # List tools
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))

In [ ]:
await ex3_list()

RESOURCES: meta=None nextCursor=None resources=[]
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Exercise 4

The conversion happens by mapping the MCP Tool's name, description, and JSON Schema (inputSchema) to the specific format expected by LLM providers (like OpenAI's `tools` array). The MCP server provides the metadata, and the client-side helper wraps it into the `type: function` structure.

In [ ]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [91]:
import asyncio
import json
import nest_asyncio
import re
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def stub_plan(prompt: str, functions: List[Dict[str, Any]]):
    """Simulate LLM tool call selection using regex."""
    prompt_lower = prompt.lower()
    calls = []
    if "add" in prompt_lower or "+" in prompt:
        nums = re.findall(r'\d+', prompt)
        if len(nums) >= 2:
            calls.append({"name": "add", "args": {"a": int(nums[0]), "b": int(nums[1])}})
    elif "multiply" in prompt_lower or "*" in prompt:
        nums = re.findall(r'\d+', prompt)
        if len(nums) >= 2:
            calls.append({"name": "multiply", "args": {"a": int(nums[0]), "b": int(nums[1])}})
    elif "greet" in prompt_lower:
        calls.append({"name": "greet", "args": {"name": "User"}})
    return calls

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls

In [88]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

            # 1. Get tools and convert
            mcp_tools = await session.list_tools()
            functions = [convert_to_llm_tool(t) for t in mcp_tools.tools]

            # 2. Get tool calls from LLM (stub or real)
            calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
            print("tool_calls:", calls)

            # 3. Execute calls
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                # Extract text content from the response
                output = [getattr(c, "text", str(c)) for c in result.content]
                print("result:", output)

In [92]:
print("--- Testing Multiply ---")
await ex5_run("Multiply 6 by 7")
print("\n--- Testing Add ---")
await ex5_run("Add 2 to 20")

--- Testing Multiply ---
tool_calls: [{'name': 'multiply', 'args': {'a': 6, 'b': 7}}]
result: ['42']

--- Testing Add ---
tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result: ['22']


## Optional - add multiply(a, b) and rerun